# Delegate to another agent

This notebook deploys `research-coordinator`, an agent that delegates web research to `web-researcher`. See the [README](./README.md) for the same steps in a terminal.

kagent and MCP servers must be enabled on your prokube cluster.

## Setup

Set up this repository's kagent helpers from [kagent_helpers.py](../shared/kagent_helpers.py). Adding their directory to `sys.path` lets the notebook import them without installing anything.

In [ ]:
import sys

sys.path.insert(0, "../shared")  # kagent helpers (shared by all kagent examples)

import kagent_helpers

In [ ]:
import os
import subprocess
from getpass import getpass
from pathlib import Path

with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
    namespace = f.read().strip()

## Prerequisites

The coordinator needs `web-researcher` from [`agent-with-mcp`](../agent-with-mcp/) and everything it depends on. The next cells create whatever is missing; `kubectl apply` leaves existing resources unchanged.

The agents call Claude with the key in the `anthropic-api-key` Secret. The next cell creates the Secret unless it already exists, for example under **K8s Secrets** in the prokube user menu (see [Kubernetes Secrets](https://prokube.ai/docs/1.9/platform/kubernetes.html#kubernetes-secrets)). It prompts for the key unless `ANTHROPIC_API_KEY` is set.

In [ ]:
secret = !kubectl get secret anthropic-api-key -o name --ignore-not-found
if secret:
    print("Using the existing anthropic-api-key Secret.")
else:
    anthropic_api_key = (os.environ.get("ANTHROPIC_API_KEY") or getpass("Anthropic API key: ")).strip()
    if not anthropic_api_key:
        raise ValueError("No Anthropic API key entered. Run the cell again and paste your key.")
    !kubectl create secret generic anthropic-api-key --from-literal=ANTHROPIC_API_KEY={anthropic_api_key}
    # The label lists the Secret under K8s Secrets in the prokube user menu.
    !kubectl label secret anthropic-api-key prokube.ai/user-secret=true

Deploy the `anthropic-haiku` ModelConfig (Claude Haiku) from the `basic-agent` example:

In [ ]:
!kubectl apply -f ../basic-agent/anthropic-model.yaml
!kubectl wait --for=condition=Accepted modelconfigs.kagent.dev/anthropic-haiku --timeout=2m

Deploy the Fetch MCP server, which fetches web pages, from the `deploy-upstream-mcp-server` example:

In [ ]:
!kubectl apply -f ../../mcp-servers/deploy-upstream-mcp-server/fetch-server.yaml
!kubectl wait --for=condition=Ready mcpservers.toolhive.stacklok.dev/fetch-example --timeout=3m

Deploy the `web-researcher` agent from the `agent-with-mcp` example. Agent Gateway prefixes tool names with the server name only when there is more than one MCP server ([tool name prefixing](https://agentgateway.dev/docs/kubernetes/latest/documentation/mcp/virtual/)):

- one server: `fetch`
- several servers: `fetch-example_fetch`

The next cell waits until `gateway-mcp` discovers the tool and writes its actual name into the agent manifest before applying it.

In [ ]:
# "fetch" with one MCP server in the workspace
# "fetch-example_fetch" with several
fetch_tool = kagent_helpers.gateway_mcp_tool_name("fetch-example", "fetch")

# The manifest lists the tool as "fetch"; swap in the name Agent Gateway uses
manifest = Path("../agent-with-mcp/web-researcher.yaml").read_text()
manifest = manifest.replace("- fetch\n", f"- {fetch_tool}\n")
subprocess.run(["kubectl", "apply", "-f", "-"], input=manifest, text=True, check=True)

!kubectl wait --for=condition=Ready agents.kagent.dev/web-researcher --timeout=3m

## Deploy

In [ ]:
!kubectl apply -f research-coordinator.yaml
!kubectl wait --for=condition=Ready agents.kagent.dev/research-coordinator --timeout=3m

## Talk to the agent from this Lab

Labs reach agents through the internal Agent Gateway route without an API key. You can also open the [Agents page](/pkui/kagent) in the prokube UI (`https://<your-prokube-domain>/pkui/kagent`), select `research-coordinator`, and chat there.

Expect one sentence about what prokube offers that includes the source URL `https://prokube.ai`, as returned by `web-researcher`.

In [ ]:
internal_url = kagent_helpers.workspace_a2a_url(namespace, "research-coordinator")
print(internal_url)

In [ ]:
prompt = "Research https://prokube.ai and summarize what prokube offers in one sentence. Include the source URL."
print(kagent_helpers.send_message(internal_url, prompt))

## Connect with an API key

Clients outside the cluster use the agent's external route, `https://<your-prokube-domain>/svc/a2a/<workspace>/research-coordinator`, with an API key. See [Public vs. Internal Traffic](https://prokube.ai/docs/1.9/platform/agent_gateway.html#public-vs-internal-traffic) for how this route differs from the internal one.

Enter your prokube domain, the host name you open prokube at, for example `my-cluster.prokube.cloud`. The cell uses `PROKUBE_DOMAIN` instead if it is set:

In [ ]:
domain = os.environ.get("PROKUBE_DOMAIN") or input("prokube domain (e.g. my-cluster.prokube.cloud): ")

Create a key with access to `research-coordinator` on the [API Keys page](/pkui/ai-gateway/keys) (`https://<your-prokube-domain>/pkui/ai-gateway/keys`), then paste it when prompted below. The cell uses `A2A_API_KEY` instead if it is set:

In [ ]:
api_key = os.environ.get("A2A_API_KEY") or getpass("Agent API key: ")

Send the same prompt through the agent's external URL:

In [ ]:
api_key_url = f"https://{domain}/svc/a2a/{namespace}/research-coordinator"
print(api_key_url)

In [ ]:
print(kagent_helpers.send_message(api_key_url, prompt, api_key=api_key))

## Clean up

Uncomment the command and run the cell to delete `research-coordinator`. This leaves `web-researcher` and its prerequisites. To remove everything, also run the clean-up cells of the earlier notebooks and delete the Fetch server.

In [ ]:
# !kubectl delete -f research-coordinator.yaml